# Get Features From Combine Tracking Data

We're checking if we can find novel insights about in-game player performance using Combine player-tracking data. We want to find new ways to examine players beyond drill outcomes such as forty yard dash time, vertical jump height, etc. 

In this file, we will try to extract features from the Combine player-tracking data that will help us predict if a wide receiver will be successful at creating separation in man coverage situations.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================

import Data_Parameters as params

<br><br/>
#### Players Data (*BDB 2027*)

In [ ]:
# =============================
# Players Data
# =============================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

# -----------------------------
# Only Keep Position of Interest
# -----------------------------

players_df = (
    players_df
    .filter(pl.col("nfl_position") == params.POSITION_OF_INTEREST)
)

# -----------------------------
# Only Keep Needed Variables
# -----------------------------
players_df = (
    players_df
    .select(
        "nfl_id", 
        "display_name", 
        "draft_year", 
        "nfl_position"
    )
)

# -----------------------------
# Identify Players of Interest
# -----------------------------

players_of_interest = (
    players_df
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()    
)

# ----------------------------
# Identify 2023 Rookie WRs
# ----------------------------

rookie_wr_2023 = (
    players_df
    .filter(pl.col("draft_year") == params.DRAFT_YEAR_OF_INTEREST)
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()
)

print("Players Data:")
print(f"    - Number of WRs From Combine: {len(players_of_interest)}")
print(f"    - Number of WRs From 2023 Rookie Class: {len(rookie_wr_2023)}")
players_df

In [ ]:
# =============================
# Combine Tracking Data
# =============================

combine_tracking_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_tracking.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(
        pl.col("nfl_id").is_in(players_of_interest)
    )
)

# -----------------------------
# Don't Keep Ball Tracking Data
# -----------------------------

combine_tracking_df = (
    combine_tracking_df
    .filter(pl.col("entity_type") == "PLAYER")
)

# ----------------------------
# Derive Velocity Variables (vx, vy)
# ----------------------------

combine_tracking_df = (
    combine_tracking_df
    .sort(["event_id", "time"])
    .with_columns(
        # Convert degrees to radians for trigonometry
        dir_rad = pl.col("dir") * (np.pi / 180)
    )
    .with_columns(
        # Reconstruct directional components of speed (velocity vectors)
        vx = pl.col("s") * pl.col("dir_rad").cos(),
        vy = pl.col("s") * pl.col("dir_rad").sin()
    )
)

# ---------------------------
# Calculate Linear Acceleration
# ---------------------------

combine_tracking_df = (
    combine_tracking_df
    # Ensure the data is strictly ordered by time within each drill event
    .sort(["event_id", "time"])
    .with_columns(
        linear_accel = (
            # (Current Speed - Previous Speed) / 0.1 seconds time delta
            (pl.col("s") - pl.col("s").shift(1).over("event_id")) / 0.1
        )
    )
)

print(f"Combine Tracking Data:")
combine_tracking_df

<br><br/>
## Choose Drills To Use
***

In [ ]:
# ==================================
# Find Percentage of Players of Interest
# With Tracking Data For Positional Drills
# ==================================

drill_tracking_summary_df = (
    combine_tracking_df
    .group_by("drill_name")
    .agg(
        num_w_atl_1_attempt = pl.col("nfl_id").n_unique()
    )
    .with_columns(
        perc_w_atl_1_attempt = (
            pl.col("num_w_atl_1_attempt")
            / combine_tracking_df.select(pl.col("nfl_id").n_unique()).item()
            * 100
        )
        .round(0)
    )
    .rename({"drill_name": "drill"})
    .sort("num_w_atl_1_attempt", descending = True)
)

print("Missingness in Position-Specific Drills:")
print(f"  - Drills With At Least 80% Participation (N = {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).height}): {drill_tracking_summary_df.filter(pl.col("perc_w_atl_1_attempt") >= 80).select("drill").to_series().to_list()}")
drill_tracking_summary_df.head(10)

In [ ]:
# ===================================
# What Drills Have Sufficient Data?
# ===================================

# From the above we see the following: 

# Non-Route Drills With Good Data
#   - Gauntlet Drill  has 100% exposure
#   - Forty Yard Dash has  88% exposure.

# Agility Drills Do NOT Have Good Data
#   - Three Cone Drill   has 23% exposure
#   - Short Shuttle Dill has 27% exposure

# The Other Drills With Good Exposure Are Drills Where The WR Is Running A Route
#   - Dagger Route Left has 100% exposure
#   - Slant Route Left  has 100% exposure
#   - Curl Route Right  has 100% exposure
#   ...

# ====================================
# Drills That Could Be Useful
# ====================================

# FORTY_YARD_DASH
#   - ~88% exposure
#   - Clear choice for measuring burst, accel, top speed

# GO_ROUTE_RIGHT
#   - 92% exposure
#   - Shows potential for measuring burst, accel, top speed

# COMEBACK_ROUTE_RIGHT
#   - ~92% exposure
#   - Shows potential for getting a decel value and cut angle

# SLOT_STRIKE_ROUTE_LEFT
#   - ~94% exposure
#   - Shows potential for getting a decel value and accelerating out of a cut

<br><br/>
## Derive Combine Performance Features
***

Here we will derive features from Combine player-tracking data. 

There are 3 types of features:
1. Acceleration  
2. Deceleration  
3. Cutting

#### Top Speed

Top Speed = Max speed player experienced during their 40-yard dash attempts

In [ ]:
# ==============================================================================
# Get Peak Velocity (Top Speed)
#   - MAX VELOCITY FEATURE
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. For Each 40-Yard Dash Attempt, Extract the Maximum Speed Achieved
# ------------------------------------------------------------------------------
forty_dash_top_speeds = (
    combine_tracking_df
    # Only look at 40-yard dash drills
    .filter(pl.col("drill_name") == "FORTY_YARD_DASH")
    # Group by individual attempt to isolate performance limits
    .group_by("event_id")
    .agg([
        pl.col("nfl_id").first(),
        # Capture the absolute maximum value from the speed column (s)
        pl.col("s").max().alias("attempt_top_speed")
    ])
    # Filter out anomalous data points or null values
    .filter(
        pl.col("attempt_top_speed").is_not_null() & 
        (pl.col("attempt_top_speed") > 0)
    )
)

# ------------------------------------------------------------------------------
# 2. For Each Player, Isolate Their Absolute Best Top Speed Across All Attempts
# ------------------------------------------------------------------------------
player_top_speed_df = (
    forty_dash_top_speeds
    .group_by("nfl_id")
    .agg([
        # Higher top speed is better, so grab the maximum across all attempts
        pl.col("attempt_top_speed").max().alias("best_top_speed")
    ])
    # Sort with the fastest players at the top of the dataframe
    .sort("best_top_speed", descending=True)
)

print(f"Top Speed Performance Stats:")
player_top_speed_df

<br><br/>
#### Zero-To-Five Time

Zero-To-Five Time = The shortest amount of time it took the player to reach a speed of 5.5 y/s (~5 m/s) in their 40-yard dash attempts

In [ ]:
# ===========================================
# Get Explosive Ratio (Zero-To-Five Time)
#   - ACCELERATION FEATURE
# ===========================================

# -------------------------------------------
# For Each 40-Yard Dash, Calculate The Zero-To-Five Time
# -------------------------------------------

forty_dash_bursts = (
    combine_tracking_df
    # Only look at 40-yard dash drills
    .filter(pl.col("drill_name") == "FORTY_YARD_DASH")
    # For each drill...
    .group_by("event_id")
    # Identify the player, when they started moving, and when they reached 5.5y/s
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        # Time when they first started moving (speed > 0.2 y/s to ignore slight tracking jitter)
        start_time = pl.col("time").filter(pl.col("s") > 0.2).min(),
        # Time when they first crossed the 5.5 y/s threshold (equivalent to ~5 m/s)
        reached_five_time = pl.col("time").filter(pl.col("s") >= 5.5).min()
    )
    # Calculate zero-to-five time
    .with_columns(
        zero_to_five_time = (pl.col("reached_five_time") - pl.col("start_time")).dt.total_milliseconds() / 1000
    )
    # Filter out invalid zero-to-five times
    .filter(
        pl.col("zero_to_five_time").is_not_null() 
        & 
        (pl.col("zero_to_five_time") > 0)
    )
)

# -----------------------------------------
# For Each Player, Get Their Best Zero-To-Five Time
# -----------------------------------------

player_zero_to_five_df = (
    forty_dash_bursts
    .group_by("nfl_id")
    .agg(
        best_zero_to_five_time = pl.col("zero_to_five_time").min()
    )
    .sort("best_zero_to_five_time", "nfl_id")
)

print(f"Zero-To-Five Time Stats:")
player_zero_to_five_df

<br><br/>
#### Peak Jerk

Peak Jerk = The player's max change in linear acceleration during any 0.1 second period (given that it's at the start of their drill, before reaching 5.5 y/s) during their 40-yard dash attempts

In [ ]:
# ===========================================
# Get Peak Jerk
#   - ACCELERATION FEATURE
# ===========================================

# -------------------------------------------
# Calculate Jerk Frame-By-Frame During 40-Yard Dash Drills
# -------------------------------------------

jerk_df = (
    combine_tracking_df
    # Only look at 40-yard dash drills
    .filter(pl.col("drill_name") == "FORTY_YARD_DASH")
    .sort(["event_id", "time"])
    # Calculate jerk: (Current Accel - Previous Accel) / 0.1 seconds
    .with_columns(
        jerk = (
            (pl.col("linear_accel") - pl.col("linear_accel").shift(1).over("event_id")) / 0.1
        )
    )
)

# -------------------------------------------
# Isolate The Initial Start Pahse &
# Find The Max Jerk Per Attempt
# -------------------------------------------

attempt_jerk_df = (
    jerk_df
    # Focus only on the initial acceleration phase (while speed is building up to 5.5 y/s)
    .filter(pl.col("s") <= 5.5)
    .group_by("event_id")
    .agg(
        pl.col("nfl_id").first(),
        max_attempt_jerk = pl.col("jerk").max()
    )
)

# ------------------------------------------
# Get Best Peak Jerk For Each Player
# ------------------------------------------

player_peak_jerk_df = (
    attempt_jerk_df
    .group_by("nfl_id")
    .agg(
        best_peak_jerk = pl.col("max_attempt_jerk").max()
    )
    .sort("best_peak_jerk", "nfl_id")
)

print("Peak Jerk Stats:")
player_peak_jerk_df

<br><br/>
#### Braking Distance & Braking Time

Braking Distance & Braking Time = The minimum amount of distance and time required to reduce their speed after beginning a hard deceleration during their COMEBACK_ROUTE_RIGHT drill attempts. 
* Start time is the first frame where `linear_accel < -3.0`
* End time is when the player reaches their minimum observed speed

In [ ]:
# ===========================================
# Get Braking Distance and Braking Time
#   - DECELERATION FEATURE
# ===========================================

# -------------------------------------------
# Identify Rows Where the Player Is Slamming On The Brakes
# -------------------------------------------

braking_events = (
    combine_tracking_df
    # Only look at COMEBACK_ROUTE_RIGHT drills
    .filter(pl.col("drill_name") == "COMEBACK_ROUTE_RIGHT")
    .sort(["event_id", "time"])
    # Flag frames where heavy braking is occurring
    .with_columns(
        is_braking = pl.col("linear_accel") < -3.0
    )
)

# -------------------------------------------
# Calculate Braking Distance & Braking Time For 
# Each Braking Event From COMEBACK_ROUTE_RIGHT Drills
# -------------------------------------------

player_braking_profile = (
    braking_events
    # --------------------------------------
    # For each event, find when braking started
    # --------------------------------------
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        # Find when braking FIRST started
        braking_start_time = pl.col("time").filter(pl.col("is_braking")).min()
    )
    # --------------------------------------
    # Add braking_events data back on
    # --------------------------------------
    # Join this start time back so we can safely filter the tracking frames chronologically
    .join(braking_events, on="event_id", how="inner")
    # --------------------------------------
    # Only keep time after braking began
    # --------------------------------------
    # Filter out all the lag frames at the start of the drill before they braked
    .filter(pl.col("time") >= pl.col("braking_start_time"))
    # --------------------------------------
    # For each event, find when the min speed occurred after braking began
    # --------------------------------------
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        braking_start_time = pl.col("braking_start_time").first(),
        # Find the minimum speed that happens AFTER braking began
        apex_time = pl.col("time").filter(pl.col("s") == pl.col("s").min()).first(),
    )
    # --------------------------------------
    # Now that we have the exact bounding timestamps, 
    # calculating braking distance and braking time
    # for each event
    # --------------------------------------
    .join(braking_events, on="event_id", how="inner")
    # Only keep time between braking and min speed apex
    .filter(
        (pl.col("time") >= pl.col("braking_start_time")) 
        & 
        (pl.col("time") <= pl.col("apex_time"))
    )
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        attempt_braking_distance_yards = pl.col("dis").sum(),
        attempt_braking_time_seconds = (pl.col("apex_time").first() - pl.col("braking_start_time").first()).dt.total_milliseconds() / 1000
    )
    .filter(pl.col("attempt_braking_time_seconds") > 0)
)

# 3. Roll up to ONE row per player, keeping their absolute BEST (minimum distance/time)

# --------------------------------------------
# Get The Best Braking Distance & Braking Time
# For Each Player
# --------------------------------------------

player_stopping_quickness_df = (
    player_braking_profile
    .group_by("nfl_id")
    .agg(
        best_braking_distance_yards = pl.col("attempt_braking_distance_yards").min(),
        best_braking_time_seconds = pl.col("attempt_braking_time_seconds").min()
    )
    .sort("best_braking_distance_yards", "best_braking_time_seconds", "nfl_id")
)

print("Braking Distance and Braking Time Stats:")
player_stopping_quickness_df

<br><br/>
#### Deceleration-To-Velocity Ratio

Deceleration-To-Velocity Ratio = The maximum decel-to-velocity ratio a player experienced during braking in any of their COMEBACK_ROUTE_RIGHT drill attempts.
* Decel-to-velocity ratio measures how large a player's peak deceleration is relative to the speed they were traveling when braking began. 
* It should capture how aggresively a player can hit the brakes relative to their entry speed.
* We require speed to be > 4.0 y/s and linear acceleration be < -3 y/s^2 in order for a decel-to-velocity ratio to be taken.

In [ ]:
# ===========================================
# Get Deceleration-To-Velocity Ratio
#   - DECELERATION FEATURE
# ===========================================

# -------------------------------------------
# Identify Time Where Player Is Braking &
# Get Their Speed Just Before Braking
# -------------------------------------------

braking_entry = (
    combine_tracking_df
    # Only look at COMEBACK_ROUTE_RIGHT drills
    .filter(pl.col("drill_name") == "COMEBACK_ROUTE_RIGHT")
    .sort(["event_id", "time"])
    .with_columns(
        # Must be braking hard AND traveling fast enough to be on an actual route
        is_true_braking = (
            (pl.col("linear_accel") < -3.0)
            & 
            (pl.col("s") > 4.0)
        ),
        # Snag the speed from the frame right before braking started
        entry_speed = pl.col("s").shift(1).over("event_id")
    )
)

# -------------------------------------------
# Get Speed At Start of Braking & Max Decel In Braking Period.
# Using That, Calculate Deceleration-To-Velocity Ratio For Event
# -------------------------------------------

attempt_ratio_df = (
    braking_entry
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        # Identify the true entry speed at the exact onset of downfield braking
        attempt_entry_speed = pl.col("entry_speed").filter(pl.col("is_true_braking")).first(),
        # Get the absolute value of the sharpest deceleration (min linear_accel) during true braking
        peak_decel_magnitude = pl.col("linear_accel").filter(pl.col("is_true_braking")).min().abs()
    )
    # Filter out any drill attempts where a true downfield break wasn't found/completed
    .filter(
        (pl.col("attempt_entry_speed").is_not_null()) 
        & 
        (pl.col("peak_decel_magnitude").is_not_null())
    )
    # Compute the ratio
    .with_columns(
        decel_to_velocity_ratio = (pl.col("peak_decel_magnitude") / pl.col("attempt_entry_speed"))
    )
)

# -------------------------------------------
# Calculate The Best Deceleration-To-Velocity
# Ratio For Each Player
# -------------------------------------------

player_decel_to_velocity_ratio_df = (
    attempt_ratio_df
    .group_by("nfl_id")
    .agg(
        best_decel_to_velocity_ratio = pl.col("decel_to_velocity_ratio").max()
    )
    .sort("best_decel_to_velocity_ratio")
)

print("Deceleration-To-Velocity Ratio:")
player_decel_to_velocity_ratio_df

<br><br/>
#### Braking Distance Per Unit of Entry Velocity

Braking Distance Per Unit of Entry Velocity = Minimum distance traveled while braking, relative to how fast they were moving when braking began. This could come from any of their COMEBACK_ROUTE_RIGHT attempts.
* Lower is better because we want the player to be able to stop quickly when they brake. 
* Period begins when linear acceleration is < -3 (while speed is > 4 y/s). 
* Period ends when minimum speed observed after braking begins.

In [ ]:
# ===========================================
# Get Braking Distance per Unit of Entry Velocity
#   - DECELERATION FEATURE
# ===========================================

# -------------------------------------------
# Identify Valid Heavy Braking Events Downfield
# -------------------------------------------

braking_base = (
    combine_tracking_df
    # Only look at COMEBACK_ROUTE_RIGHT drills
    .filter(pl.col("drill_name") == "COMEBACK_ROUTE_RIGHT")
    .sort(["event_id", "time"])
    .with_columns(
        # Player must be braking hard AND traveling fast enough to be on an actual route
        is_true_braking = (
            (pl.col("linear_accel") < -3.0) 
            & 
            (pl.col("s") > 4.0)
        ),
        # Grab the speed from the frame right before braking started
        entry_speed = pl.col("s").shift(1).over("event_id")
    )
)

# --------------------------------------------
# Extract Bounding Timestamps and Entry Speed Per Attempt
# --------------------------------------------

attempt_bounds = (
    braking_base
    # ------------------------------------------
    # For Each Event, Get Time When Braking Event Starts
    # And Speed When Event Started
    # ------------------------------------------
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        braking_start_time = pl.col("time").filter(pl.col("is_true_braking")).min(),
        attempt_entry_speed = pl.col("entry_speed").filter(pl.col("is_true_braking")).first(),
    )
    # ------------------------------------------
    # Look At Data After Brake Event Begins
    # ------------------------------------------
    .join(braking_base, on="event_id", how="inner")
    .filter(pl.col("time") >= pl.col("braking_start_time"))
    # ------------------------------------------
    # Group Again To Find The Post-Braking Lowest Speed Apex
    # ------------------------------------------
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        braking_start_time = pl.col("braking_start_time").first(),
        attempt_entry_speed = pl.col("attempt_entry_speed").first(),
        apex_time = pl.col("time").filter(pl.col("s") == pl.col("s").min()).first(),
    )
)

# --------------------------------------------
# Calculate Distance Covered And Compute The Final Fused Metric
# --------------------------------------------

player_fused_stopping_df = (
    attempt_bounds
    .join(braking_base, on="event_id", how="inner")
    # Only keep time between brake start and min speed apex
    .filter(
        (pl.col("time") >= pl.col("braking_start_time")) 
        & 
        (pl.col("time") <= pl.col("apex_time"))
    )
    .group_by("event_id")
    .agg(
        nfl_id = pl.col("nfl_id").first(),
        attempt_entry_speed = pl.col("attempt_entry_speed").first(),
        attempt_braking_distance_yards = pl.col("dis").sum(),
        attempt_braking_time_seconds = pl.col("apex_time").first() - pl.col("braking_start_time").first()
    )
    # Filter out invalid attempts
    .filter(
        (pl.col("attempt_braking_time_seconds") > 0) 
        & 
        (pl.col("attempt_entry_speed") > 0)
    )
    # Combine distance and entry speed into our fused metric
    .with_columns(
        braking_distance_per_entry_velocity = pl.col("attempt_braking_distance_yards") / pl.col("attempt_entry_speed")
    )
    # Roll up to ONE row per player, keeping their absolute lowest (best) efficiency score
    .group_by("nfl_id")
    .agg(
        best_braking_distance_per_entry_velocity = pl.col("braking_distance_per_entry_velocity").min(),
        associated_braking_time_seconds = (pl.col("attempt_braking_time_seconds").filter(pl.col("braking_distance_per_entry_velocity") == pl.col("braking_distance_per_entry_velocity").min()).first()).dt.total_milliseconds() / 1000
    )
    .sort("best_braking_distance_per_entry_velocity", "nfl_id")
)

print("Braking Distance per Unit of Entry Velocity:")
player_fused_stopping_df

<br><br/>
#### Velocity Maintenance Factor (VMF)

VMF = The max value of VMF across all SLOT_STRIKE_ROUTE_LEFT drill attempts, where VMF is the measure of how much speed a player retains immediately after their sharpest braking moment, relative to their speed immediately before it.

In [ ]:
# ===========================================
# Get Velocity Maintenance Factor (VMF)
#   - CUTTING FEATURE
# ===========================================

# -------------------------------------------
# Identify the Single True Sharp Braking Moment Per Event
# -------------------------------------------

true_apex_frames = (
    combine_tracking_df
    .filter(
        (pl.col("drill_name") == "SLOT_STRIKE_ROUTE_LEFT") & 
        (pl.col("s") > 4.0) # Player must be traveling at a functional route speed
    )
    .sort(["event_id", "time"])
    .group_by("event_id")
    .agg([
        pl.col("nfl_id").first(),
        # Anchor the apex time to when they slam on the brakes the hardest
        pl.col("time").filter(pl.col("linear_accel") == pl.col("linear_accel").min()).first().alias("apex_time")
    ])
)

# -------------------------------------------
# Calculate Entry and Exit Speeds Relative to True Apex
# -------------------------------------------

attempt_vmf_fixed_df = (
    combine_tracking_df
    .filter(pl.col("drill_name") == "SLOT_STRIKE_ROUTE_LEFT")
    .join(true_apex_frames, on="event_id", how="inner")
    .sort(["event_id", "time"])
    .group_by("event_id")
    .agg([
        pl.col("nfl_id").first(),
        # FIXED: Use pl.duration to safely perform datetime subtraction/addition
        pl.col("s").filter(pl.col("time") == (pl.col("apex_time") - pl.duration(seconds=0.2))).first().alias("v_entry"),
        pl.col("s").filter(pl.col("time") == (pl.col("apex_time") + pl.duration(seconds=0.2))).first().alias("v_exit")
    ])
    .filter(
        (pl.col("v_entry").is_not_null()) 
        & 
        (pl.col("v_entry") > 1.0) 
        & 
        (pl.col("v_exit").is_not_null())
    )
    .with_columns(
        vmf = pl.col("v_exit") / pl.col("v_entry")
    )
)

# --------------------------------------------
# Get Best VMF For Each Player
# --------------------------------------------

player_vmf_df = (
    attempt_vmf_fixed_df
    .group_by("nfl_id")
    .agg(
        best_vmf = pl.col("vmf").max()
    )
    .sort("best_vmf", "nfl_id")
)


print("Velocity Maintenance Factor (VMF):")
player_vmf_df

<br><br/>
#### Best Cut Angle

Best Cut Angle = The max change in direction over a short window for a player across all their attempts of the SLOT_STRIKE_ROUTE_LEFT drill.

In [ ]:
# ===========================================
# Get Kinematic Deviation Angle (Cut Sharpness)
#   - CUTTING FEATURE
# ===========================================

# -------------------------------------------
# Find Frames Where The Player Is Cutting Hard
# -------------------------------------------

cutting_base = (
    combine_tracking_df
    .filter(pl.col("drill_name") == "SLOT_STRIKE_ROUTE_LEFT")
    .sort(["event_id", "time"])
    .with_columns(
        had_heavy_brake = pl.col("linear_accel") < -3.0
    )
)

# -------------------------------------------
# Find The True Cut Apex And Get Entry/Exit
# Angles Shifted By 0.2 Seconds (2 Frames)
# -------------------------------------------

attempt_angle_df = (
    cutting_base
    .sort(["event_id", "time"])
    .with_columns(
        # Identify entry direction (2 frames before the current frame)
        dir_minus_0_2s = pl.col("dir").shift(2).over("event_id"),
        # Identify exit direction (2 frames after the current frame)
        dir_plus_0_2s = pl.col("dir").shift(-2).over("event_id"),
        # Filter check to see if a hard brake happened recently
        is_valid_cut = (
            pl.col("had_heavy_brake")
            .cast(pl.Int32)
            .rolling_sum(window_size=5, center=True)
            .over("event_id") > 0
        )
    )
    # --------------------------------------------
    # Filter for the exact apex frame where speed is at its local minimum during a valid cut
    # --------------------------------------------
    .filter(
        (pl.col("s") == pl.col("s").rolling_min(window_size=5, center=True).over("event_id")) 
        & 
        (pl.col("is_valid_cut") == True) 
        &
        (pl.col("s") > 0.5)
    )
    # --------------------------------------------
    # Grab the initial entry and exit angles for the sharpest cut apex per event
    # --------------------------------------------
    .group_by("event_id")
    .agg([
        pl.col("nfl_id").first(),
        pl.col("dir_minus_0_2s").filter(pl.col("s") == pl.col("s").min()).first().alias("angle_entry"),
        pl.col("dir_plus_0_2s").filter(pl.col("s") == pl.col("s").min()).first().alias("angle_exit")
    ])
    # --------------------------------------------
    # Filter out missing values
    # --------------------------------------------
    .filter(
        (pl.col("angle_entry").is_not_null()) 
        & 
        (pl.col("angle_exit").is_not_null())
    )
    # --------------------------------------------
    # Calculate the raw absolute difference and correct for 360-degree wrapping
    # --------------------------------------------
    .with_columns(
        raw_diff = (pl.col("angle_exit") - pl.col("angle_entry")).abs()
    )
    .with_columns(
        cut_angle_degrees = pl.when(pl.col("raw_diff") > 180.0)
                            .then(360.0 - pl.col("raw_diff"))
                            .otherwise(pl.col("raw_diff"))
    )
)

attempt_angle_df

# --------------------------------------------
# Get Best (Sharpest/Maximum) Cut Angle For Each Player
# --------------------------------------------

player_cut_angle_df = (
    attempt_angle_df
    .group_by("nfl_id")
    .agg(
        best_cut_angle = pl.col("cut_angle_degrees").max()
    )
    .sort("best_cut_angle", "nfl_id")
)

print("Cut Angle:")
player_cut_angle_df

<br><br/>
#### Burst Out Of Cut (BOOC)

BOOC = The max measure of BOOC across all attempts of the SLOT_STRIKE_ROUTE_LEFT drill, where BOOC measures the player's average acceleration immediately after a cut movement. This quanitfies the player's ability to explosively recaccelerate after braking and redirecting.

* Higher values are better. Higher values mean faster speed recovery after a cut.

In [ ]:
# ===========================================
# Get Burst Out of Cut (BOOC)
#   - CUTTING FEATURE
# ===========================================

# -------------------------------------------
# Find Frames Where The Player Is Cutting Hard
# -------------------------------------------

cutting_base = (
    combine_tracking_df
    .filter(pl.col("drill_name") == "SLOT_STRIKE_ROUTE_LEFT")
    .sort(["event_id", "time"])
    .with_columns(
        had_heavy_brake = pl.col("linear_accel") < -3.0
    )
)

# -------------------------------------------
# Find The True Cut Apex And Get The Average
# Acceleration Over The Next 0.3 Seconds (3 Frames)
# -------------------------------------------

attempt_booc_df = (
    cutting_base
    .sort(["event_id", "time"])
    .with_columns(
        # FIXED: Look ahead by shifting -3 frames to bring future frames backward,
        # then calculate the rolling average to perfectly grab the 3 frames AFTER the apex
        booc_rolling = (
            pl.col("linear_accel")
            .shift(-3)
            .over("event_id")
            .rolling_mean(window_size=3)
        ),
        # Filter check to see if a hard brake happened recently
        is_valid_cut = (
            pl.col("had_heavy_brake")
            .cast(pl.Int32)
            .rolling_sum(window_size=5, center=True)
            .over("event_id") > 0
        )
    )
    # --------------------------------------------
    # Filter for the exact apex frame where speed is at its local minimum during a valid cut
    # --------------------------------------------
    .filter(
        (pl.col("s") == pl.col("s").rolling_min(window_size=5, center=True).over("event_id")) & 
        (pl.col("is_valid_cut") == True) &
        (pl.col("s") > 0.5)
    )
    # --------------------------------------------
    # Grab the rolling BOOC value associated with the sharpest cut apex per event
    # --------------------------------------------
    .group_by("event_id")
    .agg([
        pl.col("nfl_id").first(),
        pl.col("booc_rolling").filter(pl.col("s") == pl.col("s").min()).first().alias("attempt_booc")
    ])
    # Filter out missing values
    .filter(pl.col("attempt_booc").is_not_null())
)

# --------------------------------------------
# Get Best (Maximum) Burst Out of Cut For Each Player
# --------------------------------------------

player_booc_df = (
    attempt_booc_df
    .group_by("nfl_id")
    .agg(
        best_booc = pl.col("attempt_booc").max()
    )
    .sort("best_booc", "nfl_id")
)

print("Burst Out Of Cut:")
player_booc_df

<br><br/>
#### Combine All Metrics Into One Dataset

In [ ]:
# =======================================
# Dataset of All Combine Metrics
# =======================================

# Join All Tables Created Above So That We Have One 
# Table With All Combine Metrics For Each Player

combine_features_df = (
    players_df
    .select("nfl_id", "display_name")
    .join(player_top_speed_df, on = "nfl_id", how = "left")
    .join(player_zero_to_five_df, on = "nfl_id", how = "left")
    .join(player_peak_jerk_df, on = "nfl_id", how = "left")
    .join(player_stopping_quickness_df, on = "nfl_id", how = "left")
    .join(player_decel_to_velocity_ratio_df, on = "nfl_id", how = "left")
    .join(player_fused_stopping_df, on = "nfl_id", how = "left")
    .join(player_vmf_df, on = "nfl_id", how = "left")
    .join(player_cut_angle_df, on = "nfl_id", how = "left")
    .join(player_booc_df, on = "nfl_id", how = "left")
)

print("Combine Features Dataset:")
print(f"    - Dimensions: {combine_features_df.shape}")
combine_features_df

<br><br/>
#### Aggregate Metrics Into A Score

Acceleration, deceleration, and cutting scores are calculated based on how a WR ranks (based on percentiles compared to their peers) in each individual metric. 

For example, if a player is 90th percentile at zero-to-five time and 70th percentile at peak jerk, then their acceleration score is 80. Which means that their average normalized rank across the acceleration metrics is the 80th percentile. 

*NOTE: The scores are relative to the population, not absolute athletic measurements. A score of X means something different if the reference population changes.*

`accel_score`
* Measures a player's ability to build speed quickly, combining how quickly they reach 5 y/s with how abruptly their acceleration changes early in the sprint

`decel_score`
* Measures the ability to generate strong negative acceleration relative to entry speed and to stop over a short distance relative to that speed

`cut_score`
* Measures a player's ability to execute a change of direction

In [ ]:
# ===========================================
# Create Blended Master Agility Score Framework
# ===========================================

combine_features_df = (
    combine_features_df
    .with_columns(
        # -------------------------------------------
        # 1. Standardize Starting Metrics (0 to 1 Percentile)
        # -------------------------------------------
        # Lower time is better (descending rank)
        pct_start_time = (pl.col("best_zero_to_five_time").rank(descending=True) / pl.len()),
        # Higher jerk is better (ascending rank)
        pct_jerk = (pl.col("best_peak_jerk").rank(descending=False) / pl.len()),
        # Higher top speed is better (ascending rank)
        pct_top_speed = (pl.col("best_top_speed").rank(descending=False) / pl.len()),

        # -------------------------------------------
        # 2. Standardize Stopping Metrics (0 to 1 Percentile)
        # -------------------------------------------
        # Higher ratio is better (ascending rank)
        pct_decel_ratio = (pl.col("best_decel_to_velocity_ratio").rank(descending=False) / pl.len()),
        # Lower space efficiency is better (descending rank)
        pct_space_eff = (pl.col("best_braking_distance_per_entry_velocity").rank(descending=True) / pl.len()),

        # -------------------------------------------
        # 3. Standardize Cutting Metrics (0 to 1 Percentile)
        # -------------------------------------------
        # Higher VMF is better (ascending rank)
        pct_vmf = (pl.col("best_vmf").rank(descending=False) / pl.len()),
        # Higher BOOC is better (ascending rank)
        pct_booc = (pl.col("best_booc").rank(descending=False) / pl.len()),
    )
    .with_columns(
        # -------------------------------------------
        # 4. Calculate Blended Category Scores (Scaled to 0-100)
        # -------------------------------------------
        accel_score = (((pl.col("pct_start_time") + pl.col("pct_jerk")) / 2.0) * 100.0),
        decel_score = (((0.5 * pl.col("pct_decel_ratio")) + 0.5 * (pl.col("pct_space_eff"))) * 100.0),
        # decel_score = (((pl.col("pct_decel_ratio") + pl.col("pct_space_eff")) / 2.0) * 100.0),
        cut_score = (((pl.col("pct_vmf") + pl.col("pct_booc")) / 2.0) * 100.0)
    )
    .with_columns(
        # -------------------------------------------
        # 5. Composite Master Agility Score
        # -------------------------------------------
        master_agility_score = (((1/3) * pl.col("accel_score")) + ((1/3) * pl.col("decel_score")) + ((1/3) * pl.col("cut_score")))
    )
)

print("Combine Metrics Dataset With Aggregated Scores:")
print(f"    - Number of Players: {combine_features_df.select("nfl_id").n_unique()}")
combine_features_df

In [ ]:
# ===========================================
# Create Scores With Normalized Values
# Rather Than Percentile Ranks
# ===========================================

# combine_features_df = (
#     combine_features_df
#     .with_columns(
#         # -------------------------------------------
#         # 1. Normalize Acceleration Metrics (0 to 1)
#         # -------------------------------------------
#         # Lower time is better: reverse normalized value
#         norm_start_time = (
#             (pl.col("best_zero_to_five_time").max() - pl.col("best_zero_to_five_time"))
#             / (
#                 pl.col("best_zero_to_five_time").max()
#                 - pl.col("best_zero_to_five_time").min()
#             )
#         ),

#         # Higher jerk is better
#         norm_jerk = (
#             (pl.col("best_peak_jerk") - pl.col("best_peak_jerk").min())
#             / (
#                 pl.col("best_peak_jerk").max()
#                 - pl.col("best_peak_jerk").min()
#             )
#         ),

#         # Higher top speed is better
#         norm_top_speed = (
#             (pl.col("best_top_speed") - pl.col("best_top_speed").min())
#             / (
#                 pl.col("best_top_speed").max()
#                 - pl.col("best_top_speed").min()
#             )
#         ),

#         # -------------------------------------------
#         # 2. Normalize Deceleration Metrics (0 to 1)
#         # -------------------------------------------
#         # Higher ratio is better
#         norm_decel_ratio = (
#             (pl.col("best_decel_to_velocity_ratio") - pl.col("best_decel_to_velocity_ratio").min())
#             / (
#                 pl.col("best_decel_to_velocity_ratio").max()
#                 - pl.col("best_decel_to_velocity_ratio").min()
#             )
#         ),

#         # Lower braking distance per entry velocity is better
#         norm_space_eff = (
#             (pl.col("best_braking_distance_per_entry_velocity").max()
#              - pl.col("best_braking_distance_per_entry_velocity"))
#             / (
#                 pl.col("best_braking_distance_per_entry_velocity").max()
#                 - pl.col("best_braking_distance_per_entry_velocity").min()
#             )
#         ),

#         # -------------------------------------------
#         # 3. Normalize Cutting Metrics (0 to 1)
#         # -------------------------------------------
#         # Higher VMF is better
#         norm_vmf = (
#             (pl.col("best_vmf") - pl.col("best_vmf").min())
#             / (
#                 pl.col("best_vmf").max()
#                 - pl.col("best_vmf").min()
#             )
#         ),

#         # Higher BOOC is better
#         norm_booc = (
#             (pl.col("best_booc") - pl.col("best_booc").min())
#             / (
#                 pl.col("best_booc").max()
#                 - pl.col("best_booc").min()
#             )
#         ),
#     )
#     .with_columns(
#         # -------------------------------------------
#         # 4. Calculate Category Scores (Scaled 0-100)
#         # -------------------------------------------
#         norm_accel_score = (
#             (pl.col("norm_start_time") + pl.col("norm_jerk")) / 2.0 * 100.0
#         ),

#         norm_decel_score = (
#             (pl.col("norm_decel_ratio") + pl.col("norm_space_eff")) / 2.0 * 100.0
#         ),

#         norm_cut_score = (
#             (pl.col("norm_vmf") + pl.col("norm_booc")) / 2.0 * 100.0
#         ),
#     )
#     .with_columns(
#         # -------------------------------------------
#         # 5. Composite Master Agility Score (0-100)
#         # -------------------------------------------
#         norm_master_agility_score = (
#             pl.col("norm_accel_score") / 3.0
#             + pl.col("norm_decel_score") / 3.0
#             + pl.col("norm_cut_score") / 3.0
#         )
#     )
# )

# print("Combine Metrics Dataset With Aggregated Scores:")
# print(f"    - Number of Players: {combine_features_df.select('nfl_id').n_unique()}")

# combine_features_df


<br><br/>
#### Examine Missingness

In [ ]:
# ========================================
# Missingness In Combine Features Dataset
# ========================================

combine_metrics = [
    "best_zero_to_five_time", 
    "best_peak_jerk", 
    "best_top_speed",
    "best_braking_distance_yards", 
    "best_braking_time_seconds", 
    "best_decel_to_velocity_ratio",	
    "best_braking_distance_per_entry_velocity", 
    "associated_braking_time_seconds", 
    "best_vmf", 
    "best_cut_angle", 
    "best_booc"
]

print("Percentage of Players With Missing Values For Each Combine Metric:")
(
    combine_features_df
    .select(combine_metrics)
    .select(((pl.all().is_null().sum()) / players_df.height * 100).round(2))
    .transpose(include_header=True, header_name="feature")
    .rename({"column_0": "percent_missing"})
    .sort("percent_missing")
)

<br><br/>
## Export Data
***

In [ ]:
# ===================================
# Combine Features Data
# ===================================

# For all WRs in our BDB2027 Rookie Cohort, we derive some metrics
# related to their acceleration, deceleration, and cutting abilities. 
# These metrics are stored here. There's one row per player.

combine_features_df.write_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))